# Multi-Datacenter Comparison

Operators often need to compare efficiency, grid stress, and carbon impact across a
fleet of data centers. This notebook shows how to run EWIS metrics across several
sites and analyze the results in a tabular format. By the end you will have a
ranked view of every facility on three axes — grid stress index (GSI),
energy-per-token (EPT), and carbon-per-token (CPT) — plus a full engine pipeline
run on the highest-stress site.

## Define the Fleet

Below we define a synthetic fleet of five data centers spread across different
regions. Each payload mirrors the `TelemetryPayload` schema used by the EWIS
toolkit and includes grid context, weather observations, and a workload block
so that every metric can be computed without fallback assumptions.

In [ ]:
fleet = [
    {
        "datacenter_id": "DC-US-WEST",
        "region": "CAISO",
        "timestamp_utc": "2026-03-03T12:00:00Z",
        "power_mw": 32.5,
        "it_load_mw": 26.2,
        "pue": 1.24,
        "grid_capacity_mw": 52000,
        "base_grid_load_mw": 28500,
        "carbon_intensity_kgco2_per_mwh": 180,
        "ambient_temp_c": 22.1,
        "humidity_pct": 41,
        "workload": {
            "model_name": "llm-alpha",
            "energy_kwh": 18.2,
            "tokens": 9_000_000,
            "tokens_per_s": 2500,
        },
    },
    {
        "datacenter_id": "DC-US-EAST",
        "region": "PJM",
        "timestamp_utc": "2026-03-03T12:00:00Z",
        "power_mw": 48.0,
        "it_load_mw": 40.0,
        "pue": 1.20,
        "grid_capacity_mw": 85000,
        "base_grid_load_mw": 62000,
        "carbon_intensity_kgco2_per_mwh": 420,
        "ambient_temp_c": 28.5,
        "humidity_pct": 72,
        "workload": {
            "model_name": "llm-beta",
            "energy_kwh": 30.5,
            "tokens": 15_000_000,
            "tokens_per_s": 4200,
        },
    },
    {
        "datacenter_id": "DC-EU-NORTH",
        "region": "NORDPOOL",
        "timestamp_utc": "2026-03-03T12:00:00Z",
        "power_mw": 20.0,
        "it_load_mw": 17.0,
        "pue": 1.18,
        "grid_capacity_mw": 40000,
        "base_grid_load_mw": 25000,
        "carbon_intensity_kgco2_per_mwh": 45,
        "ambient_temp_c": 8.0,
        "humidity_pct": 55,
        "workload": {
            "model_name": "llm-gamma",
            "energy_kwh": 12.0,
            "tokens": 8_000_000,
            "tokens_per_s": 3000,
        },
    },
    {
        "datacenter_id": "DC-APAC-SOUTH",
        "region": "NEM",
        "timestamp_utc": "2026-03-03T12:00:00Z",
        "power_mw": 55.0,
        "it_load_mw": 42.0,
        "pue": 1.31,
        "grid_capacity_mw": 60000,
        "base_grid_load_mw": 48000,
        "carbon_intensity_kgco2_per_mwh": 650,
        "ambient_temp_c": 35.2,
        "humidity_pct": 85,
        "workload": {
            "model_name": "llm-delta",
            "energy_kwh": 40.0,
            "tokens": 20_000_000,
            "tokens_per_s": 5000,
        },
    },
    {
        "datacenter_id": "DC-ME-CENTRAL",
        "region": "GCCIA",
        "timestamp_utc": "2026-03-03T12:00:00Z",
        "power_mw": 38.0,
        "it_load_mw": 28.5,
        "pue": 1.42,
        "grid_capacity_mw": 45000,
        "base_grid_load_mw": 35000,
        "carbon_intensity_kgco2_per_mwh": 550,
        "ambient_temp_c": 42.0,
        "humidity_pct": 25,
        "workload": {
            "model_name": "llm-epsilon",
            "energy_kwh": 25.0,
            "tokens": 12_000_000,
            "tokens_per_s": 3500,
        },
    },
]

print(f"{len(fleet)} data centers defined")

## Compute Metrics for Each Site

In [ ]:
from ewis.metrics.core import grid_stress_index, energy_per_token, carbon_per_token
import pandas as pd

rows = []
for payload in fleet:
    gsi_result = grid_stress_index(payload)
    ept_result = energy_per_token(payload)
    cpt_result = carbon_per_token(payload)

    rows.append(
        {
            "datacenter_id": payload["datacenter_id"],
            "region": payload["region"],
            "power_mw": payload["power_mw"],
            "pue": payload["pue"],
            "gsi": gsi_result["value"],
            "ept": ept_result["value"],
            "cpt": cpt_result["value"],
            "carbon_intensity": payload["carbon_intensity_kgco2_per_mwh"],
        }
    )

df = pd.DataFrame(rows)
df

## Ranking by Grid Stress

The Grid Stress Index (GSI) quantifies how much pressure a facility places on the
local grid relative to available headroom and environmental conditions. Higher
values indicate greater stress. A site with high GSI may benefit from load
shifting, demand response, or capacity upgrades.

In [ ]:
df_gsi = df.sort_values("gsi", ascending=False).reset_index(drop=True)
df_gsi.index += 1
df_gsi.index.name = "rank"
df_gsi[["datacenter_id", "region", "power_mw", "gsi"]]

# Sites near the top of this ranking are operating on grids with the least
# remaining headroom and/or under challenging environmental conditions.

## Carbon Efficiency Analysis

Carbon-per-token (CPT) is a compound metric: it depends on both the regional
carbon intensity of the electricity supply and the energy efficiency of the
workload itself. Two facilities with identical workloads can differ dramatically
in CPT simply because one draws from a hydro-heavy grid and the other from a
coal-heavy grid.

In [ ]:
df_cpt = df.sort_values("cpt", ascending=False).reset_index(drop=True)
df_cpt["carbon_ratio"] = df_cpt["cpt"] / df_cpt["cpt"].min()
df_cpt[["datacenter_id", "region", "carbon_intensity", "cpt", "carbon_ratio"]]

## Energy Efficiency Leaderboard

In [ ]:
df_ept = df.sort_values("ept", ascending=True).reset_index(drop=True)
df_ept.index += 1
df_ept.index.name = "rank"
df_ept[["datacenter_id", "region", "pue", "ept"]]

## Fleet Summary Statistics

In [ ]:
numeric_cols = ["power_mw", "pue", "gsi", "ept", "cpt", "carbon_intensity"]
print(df[numeric_cols].describe().to_string())
print()

total_power = df["power_mw"].sum()
avg_pue = df["pue"].mean()
best_gsi_row = df.loc[df["gsi"].idxmin()]
worst_gsi_row = df.loc[df["gsi"].idxmax()]

print(f"Total fleet power:  {total_power:.1f} MW")
print(f"Average PUE:        {avg_pue:.2f}")
print(f"Best GSI:           {best_gsi_row['gsi']:.4f}  ({best_gsi_row['datacenter_id']})")
print(f"Worst GSI:          {worst_gsi_row['gsi']:.4f}  ({worst_gsi_row['datacenter_id']})")

## Running Through the Engine

For a deeper analysis we can run a single site through the full EWIS engine
pipeline, which validates the payload, computes all metrics, and executes any
registered plugins. Below we target **DC-APAC-SOUTH** — the highest-stress
site — and include the `CoolingOptimizerPlugin` to get a cooling posture
recommendation.

In [ ]:
import json

from ewis.core.engine import EWISEngine
from ewis.plugins.builtin.cooling_optimizer import CoolingOptimizerPlugin

engine = EWISEngine()
engine.register(CoolingOptimizerPlugin())

# DC-APAC-SOUTH is at index 3 in the fleet list
apac_payload = fleet[3]
report = engine.run(apac_payload)

print(json.dumps(report.summary(), indent=2, default=str))

## Key Insights

- **Geographic location and carbon intensity dominate carbon-per-token differences.**
  A facility on a low-carbon grid (e.g., Nordic hydro) can be an order of magnitude
  cleaner per token than one on a fossil-heavy grid, even with similar hardware.
- **PUE directly affects overhead power and indirectly affects all efficiency metrics.**
  Reducing PUE shrinks the gap between total facility draw and useful IT work,
  improving both energy-per-token and the resulting carbon footprint.
- **Grid stress is a function of both facility demand and regional headroom.**
  A large facility on a large grid can have lower GSI than a smaller facility on a
  constrained grid.
- **Fleet-wide comparison helps prioritize which sites need optimization first.**
  Ranking by GSI, EPT, and CPT surfaces the facilities where targeted investment
  (cooling upgrades, workload migration, renewable procurement) yields the greatest
  return.